# Pixel3DMM — Milestone 1 Geometry Bake-Off (Colab A100)

Hair App 첫 3D 실험: 사용자 다중 사진에서 **hairless head mesh**를 재현하고 3D로 확인한다.

- 계획: `experiments/milestone1_geometry_bakeoff/README.md`
- source of truth: `docs/10_3d_hair_app_master_plan.md` (Milestone 1)
- **License: CC BY-NC 4.0 (비상업 연구). 상용 가능 아님.**

> ⚠️ **Privacy:** private 사진/출력은 절대 git에 넣지 않는다. Google Drive `MyDrive/hair_app` 에만 둔다.
> ⚠️ **셀을 위에서 아래로 하나씩** 실행한다. condacolab 셀은 커널을 재시작한다(정상).

### 검증 상태 (2026-06-21)
- ✅ 환경 빌드(pytorch3d/nvdiffrast) ~ 전처리(cropping+segmentation)까지 실제로 통과 확인
- ⚠️ network_inference / tracking / FLAME 은 아직 end-to-end 검증 전 (특히 FLAME 다운로드 주의)

## 0. GPU 확인 (A100 기대)

In [ ]:
!nvidia-smi

## 1. conda 설치 (condacolab)

이 셀 실행 후 **커널이 자동 재시작**된다. 재시작되면 다음 셀부터 이어서 실행한다.

In [ ]:
!pip -q install condacolab
import condacolab
condacolab.install()  # 커널 재시작됨 (정상)

## 2. 저장소 clone

In [ ]:
import condacolab; condacolab.check()
import os
%cd /content
if not os.path.exists('/content/pixel3dmm'):
    !git clone https://github.com/SimonGiebenhain/pixel3dmm.git
%cd /content/pixel3dmm
!git rev-parse HEAD   # 재현용 commit 기록

## 3. 환경 생성 (conda env + torch + **CUDA 11.8 toolkit**)

공식 README manual 경로. **A100이므로 `TORCH_CUDA_ARCH_LIST="8.0+PTX"`**.
(H100=`9.0+PTX`, T4=`7.5+PTX` 로 바꿀 것.)

⚠️ pytorch3d/nvdiffrast는 **CUDA toolkit(nvcc)** 가 env에 있어야 컴파일된다. 빌드는 10~25분 걸린다.

### 3-1. conda env 생성

In [ ]:
%%bash
set -e
source activate base || true
conda create -n p3dmm python=3.9 -y
# 이후 셀은 `conda run -n p3dmm ...` 로 실행 (Colab은 conda activate 지속이 어려움)

### 3-2. torch (cu118)

In [ ]:
%%bash
set -e
conda run -n p3dmm pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118

### 3-3. CUDA 11.8 toolkit (nvcc + dev libs) — 공식 README 명령

In [ ]:
%%bash
set -e
conda install -n p3dmm -y \
  nvidia/label/cuda-11.8.0::cuda-nvcc       nvidia/label/cuda-11.8.0::cuda-cccl \
  nvidia/label/cuda-11.8.0::cuda-cudart     nvidia/label/cuda-11.8.0::cuda-cudart-dev \
  nvidia/label/cuda-11.8.0::libcusparse     nvidia/label/cuda-11.8.0::libcusparse-dev \
  nvidia/label/cuda-11.8.0::libcublas       nvidia/label/cuda-11.8.0::libcublas-dev \
  nvidia/label/cuda-11.8.0::libcurand       nvidia/label/cuda-11.8.0::libcurand-dev \
  nvidia/label/cuda-11.8.0::libcusolver     nvidia/label/cuda-11.8.0::libcusolver-dev
conda run -n p3dmm nvcc --version   # 'release 11.8' 보이면 정상

### 3-4. pytorch3d / nvdiffrast 소스 빌드 (10~25분 소요)

In [ ]:
%%bash
set -e
ENVDIR=/usr/local/envs/p3dmm
export CUDA_HOME=$ENVDIR
export TORCH_CUDA_ARCH_LIST="8.0+PTX"   # A100
conda run -n p3dmm pip install ninja fvcore iopath
CUDA_HOME=$ENVDIR TORCH_CUDA_ARCH_LIST="8.0+PTX" \
  conda run -n p3dmm pip install --no-build-isolation "git+https://github.com/facebookresearch/pytorch3d.git@stable"
CUDA_HOME=$ENVDIR TORCH_CUDA_ARCH_LIST="8.0+PTX" \
  conda run -n p3dmm pip install --no-build-isolation "git+https://github.com/NVlabs/nvdiffrast.git"

### 3-5. Pixel3DMM 의존성 + 패키지 설치

In [ ]:
%%bash
set -e
cd /content/pixel3dmm
conda run -n p3dmm pip install -r requirements.txt
conda run -n p3dmm pip install -e .

## 4. 전처리 파이프라인 설치 (facer / PIPNet) + 가중치

⚠️ 공식 `install_preprocessing_pipeline.sh`는 **SSH(git@github.com)** clone을 써서 Colab에서 실패한다.
그래서 아래처럼 **HTTPS로 직접 설치**한다. 또 FaceBoxes 빌드에 **Cython**이 필요하다.
이 셀이 facer/PIPNet/가중치(uv·normals ckpt)를 올바른 위치에 받는다.

### 4-1. facer + PIPNet + ckpt (HTTPS, Cython 빌드)

In [ ]:
%%bash
set -e
PRE=/content/pixel3dmm/src/pixel3dmm/preprocessing
conda run -n p3dmm pip install -q Cython gdown

# 잔여물 정리 (재실행 안전)
rm -rf "$PRE/facer" "$PRE/MICA" "$PRE/PIPNet" \
       /content/pixel3dmm/src/PIPNet /content/pixel3dmm/src/MICA /content/pixel3dmm/src/facer
cd "$PRE"

# 1) facer (segmentation) — SSH -> HTTPS
git clone https://github.com/FacePerceiver/facer.git
cd facer
cp ../replacement_code/farl.py facer/face_parsing/farl.py
cp ../replacement_code/facer_transform.py facer/transform.py
conda run -n p3dmm pip install -e .
cd "$PRE"

# 2) PIPNet (cropping/landmarks) — 올바른 위치 + Cython 빌드 + 가중치
git clone https://github.com/jhb86253817/PIPNet.git
cd PIPNet/FaceBoxesV2/utils
conda run -n p3dmm sh make.sh
cd "$PRE/PIPNet"
mkdir -p snapshots/WFLW/pip_32_16_60_r18_l2_l1_10_1_nb10/
conda run -n p3dmm gdown 1nVkaSbxy3NeqblwMTGvLg4nF49cI_99C \
  -O snapshots/WFLW/pip_32_16_60_r18_l2_l1_10_1_nb10/epoch59.pth

# 3) MICA (best-effort, HTTPS; tracking은 ignore_mica=True 라 없어도 됨)
cd "$PRE"
git clone https://github.com/Zielon/MICA.git || true

# 4) normals/uv 예측 체크포인트 (올바른 위치)
mkdir -p /content/pixel3dmm/pretrained_weights
cd /content/pixel3dmm/pretrained_weights
conda run -n p3dmm gdown 1SDV_8_qWTe__rX_8e4Fi-BE3aES0YzJY -O ./uv.ckpt
conda run -n p3dmm gdown 1KYYlpN-KGrYMVcAOT22NkVQC0UAfycMD -O ./normals.ckpt
echo DONE-PRE-INSTALL

### 4-2. facer segmentation 패치 (torch 호환)

torch 최신 버전에서 `farl.py`의 인덱싱이 `IndexError: tensors used as indices must be long...` 로 실패한다.
인덱스를 `.long()`으로 캐스팅해 고친다. **4-1을 다시 돌렸으면 이 셀도 다시 실행**(cp가 원본을 덮어씀).

In [ ]:
from pathlib import Path
p = Path('/content/pixel3dmm/src/pixel3dmm/preprocessing/facer/facer/face_parsing/farl.py')
t = p.read_text()
t = t.replace("images[data['image_ids']]", "images[data['image_ids'].long()]")
p.write_text(t)
print('patched farl.py ->', "images[data['image_ids'].long()]" in t)

### 4-3. FLAME 다운로드 ⚠️ 미검증

tracking(9번)에는 FLAME 모델이 필요하다. 공식 `download_flame2023.sh`를 쓰지만,
**FLAME은 https://flame.is.tue.mpg.de 계정/동의가 필요**하고 Colab에서 7997바이트짜리 HTML만 받아지는 경우가 있었다.
이 단계는 아직 end-to-end로 검증되지 않았다. 9번에서 FLAME 관련 에러가 나면 여기를 다시 점검한다.

In [ ]:
%%bash
set -e
cd /content/pixel3dmm
conda run -n p3dmm ./download_flame2023.sh || echo 'FLAME download may need attention (see 4-3 note)'

## 5. 환경 변수 파일 작성

`~/.config/pixel3dmm/.env` 에 경로 지정. tracking 출력은 로컬(빠름) → 이후 Drive로 복사.

In [ ]:
import pathlib
cfg = pathlib.Path.home() / '.config' / 'pixel3dmm'
cfg.mkdir(parents=True, exist_ok=True)
env = (
    'PIXEL3DMM_CODE_BASE="/content/pixel3dmm"\n'
    'PIXEL3DMM_PREPROCESSED_DATA="/content/p3dmm_preprocessed"\n'
    'PIXEL3DMM_TRACKING_OUTPUT="/content/p3dmm_tracking"\n'
)
(cfg / '.env').write_text(env)
print((cfg / '.env').read_text())

## 6. private 입력 준비 (Google Drive `hair_app`)

Drive를 마운트하고 `MyDrive/hair_app/inputs/` 폴더에 본인 사진을 넣는다.
**Pixel3DMM은 폴더 안의 모든 이미지를 읽으므로 파일명은 자유다.**
`VID_NAME`은 **입력 폴더 이름에서 자동 도출**한다(전처리 출력 폴더명과 항상 일치하도록).

In [ ]:
from google.colab import drive
import os
drive.mount('/content/drive')

INPUT_PATH = '/content/drive/MyDrive/hair_app/inputs'   # 이 폴더에 사진을 넣으면 됨 (파일명 자유)
VID_NAME = os.path.basename(INPUT_PATH.rstrip('/'))     # = 'inputs' (전처리 출력 폴더명과 일치)
os.makedirs(INPUT_PATH, exist_ok=True)
imgs = [f for f in os.listdir(INPUT_PATH) if f.lower().endswith(('.jpg','.jpeg','.png'))]
print('input 폴더:', INPUT_PATH, '| VID_NAME:', VID_NAME)
print('이미지 개수:', len(imgs), imgs[:10])
assert imgs, 'inputs 폴더가 비어있음 → Drive/hair_app/inputs 에 사진을 넣어주세요'

## 7. Step 1 — 전처리 (crop / landmark / segmentation)

In [ ]:
%%bash -s "$INPUT_PATH"
cd /content/pixel3dmm
conda run -n p3dmm python scripts/run_preprocessing.py --video_or_images_path "$1"

### 7-1. 전처리 결과 확인

MICA 관련 에러(`can't cd to ...MICA`)는 무시해도 된다(tracking에서 ignore).

In [ ]:
import glob
print('cropped:', len(glob.glob(f'/content/p3dmm_preprocessed/{VID_NAME}/cropped/*')))
print('seg:', len(glob.glob(f'/content/p3dmm_preprocessed/{VID_NAME}/seg*/*')))
print('ckpts:', glob.glob('/content/pixel3dmm/pretrained_weights/*.ckpt'))

## 8. Step 2 — 네트워크 추론 (normals, uv_map)

In [ ]:
%%bash -s "$VID_NAME"
cd /content/pixel3dmm
conda run -n p3dmm python scripts/network_inference.py model.prediction_type=normals video_name="$1"
conda run -n p3dmm python scripts/network_inference.py model.prediction_type=uv_map video_name="$1"

## 9. Step 3 — 트래킹 (multi-image 모드) ⚠️ FLAME 필요

다중 사진 identity fusion. 단일 이미지면 `... track.py video_name=$VID_NAME iters=800`.
FLAME(4-3)이 제대로 안 받아졌으면 여기서 막힐 수 있다.

In [ ]:
%%bash -s "$VID_NAME"
cd /content/pixel3dmm
conda run -n p3dmm python scripts/track.py video_name="$1" \
  iters=100 iters=1500 include_neck=False use_flame2023=True ignore_mica=True is_discontinuous=True

## 10. ✅ 3D 결과 미리보기 (인터랙티브, 마우스로 회전)

트래킹이 만든 head mesh를 노트북 안에서 바로 돌려본다. 이 단계 결과는 **회색 무텍스처 민머리 두상**이다
(피부 텍스처=Milestone 2, 머리카락=Milestone 4 는 아직 아님).

In [ ]:
!pip -q install trimesh plotly
import glob, trimesh
import plotly.graph_objects as go

cands = []
for ext in ('ply', 'obj'):
    cands += glob.glob(f'/content/p3dmm_tracking/**/*.{ext}', recursive=True)
print('찾은 mesh 개수:', len(cands))
for p in sorted(cands)[-10:]:
    print(p)
assert cands, 'mesh 없음 → 9번 트래킹 셀이 정상 종료됐는지, 출력 경로가 맞는지 확인'

mesh_path = sorted(cands)[-1]   # 가장 최근 것. 위 목록에서 골라 교체 가능
m = trimesh.load(mesh_path, force='mesh')
v, f = m.vertices, m.faces
fig = go.Figure(data=[go.Mesh3d(
    x=v[:, 0], y=v[:, 1], z=v[:, 2],
    i=f[:, 0], j=f[:, 1], k=f[:, 2],
    color='lightgray', flatshading=True)])
fig.update_layout(scene=dict(aspectmode='data'), margin=dict(l=0, r=0, t=0, b=0))
fig.show()
print('표시한 mesh:', mesh_path)

## 11. 결과/manifest를 Drive(`hair_app`)에 저장 (재현성 + 영속화)

Colab runtime은 ephemeral이므로 mesh와 run manifest를 Drive로 복사한다.

In [ ]:
import json, subprocess, datetime, pathlib, glob, shutil

commit = subprocess.run(['git','-C','/content/pixel3dmm','rev-parse','HEAD'],
                        capture_output=True, text=True).stdout.strip()
manifest = {
    'model': 'pixel3dmm',
    'commit': commit,
    'license': 'CC BY-NC 4.0 (non-commercial)',
    'gpu': 'A100',
    'torch_cuda_arch_list': '8.0+PTX',
    'input_set_id': VID_NAME,
    'tracking_config': 'iters=100 iters=1500 include_neck=False use_flame2023=True ignore_mica=True is_discontinuous=True',
    'created_at': datetime.datetime.utcnow().isoformat() + 'Z',
    'fixes_applied': [],   # TODO: 실제 적용한 수정 기록
}

base = pathlib.Path('/content/drive/MyDrive/hair_app')
(base / 'manifests').mkdir(parents=True, exist_ok=True)
(base / 'results').mkdir(parents=True, exist_ok=True)
(base / 'manifests' / f'pixel3dmm_{VID_NAME}.json').write_text(
    json.dumps(manifest, indent=2, ensure_ascii=False))

copied = 0
for ext in ('ply', 'obj'):
    for p in glob.glob(f'/content/p3dmm_tracking/**/*.{ext}', recursive=True):
        shutil.copy(p, base / 'results' / pathlib.Path(p).name)
        copied += 1
print(f'manifest + mesh {copied}개를 {base} 에 저장')
print(json.dumps(manifest, indent=2, ensure_ascii=False))

## 12. 점수화

`scoring_sheet.csv`에 identity/geometry/hairline/side_contour/scalp_ear_topology/
execution_reliability(1–5)를 기록. hidden scalp/rear는 측정값이 아니라 prior 추정임을 유의.